# AG03 - Memory Systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nexageapps/AI/blob/main/Agents/AG03%20-%20Memory%20Systems.ipynb)

**What you'll learn:**
- Why agents need memory
- Conversation buffer memory
- Conversation summary memory
- Entity memory and knowledge graphs
- Vector store memory (semantic search)
- Memory persistence strategies
- Choosing the right memory type

**Duration:** 2 hours

---

## 1. Why Do Agents Need Memory?

### The Problem: Stateless LLMs

LLMs by themselves have **no memory** of previous interactions:

```
User: "My name is Alice"
LLM: "Hello Alice!"

[New conversation]
User: "What's my name?"
LLM: "I don't know your name."  ❌
```

### The Solution: Memory Systems

Memory systems store and retrieve context:

```
User: "My name is Alice"
Agent: [stores in memory: name="Alice"]
Agent: "Hello Alice!"

User: "What's my name?"
Agent: [retrieves from memory: name="Alice"]
Agent: "Your name is Alice!"  ✅
```

### Types of Memory

1. **Buffer Memory** - Stores recent conversation turns
2. **Summary Memory** - Compresses old messages into summaries
3. **Entity Memory** - Tracks specific entities (people, places)
4. **Vector Memory** - Semantic search over past conversations
5. **Knowledge Graph** - Structured relationships between entities

## 2. Setup

In [ ]:
# Install dependencies
!pip install -qU \
    langchain \
    langchain-openai \
    langchain-community \
    chromadb \
    faiss-cpu \
    python-dotenv

print("✅ Installation complete!")

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# Initialize LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

print("✅ Setup complete")

## 3. Conversation Buffer Memory

Stores all messages in a buffer (list of messages).

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

# Create memory
memory = ConversationBufferMemory(
    return_messages=True,  # Return as message objects
)

# Create conversation chain
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True  # Show what's happening
)

# Have a conversation
print("Turn 1:")
print(conversation.predict(input="Hi, I'm learning about AI agents"))

print("\nTurn 2:")
print(conversation.predict(input="What was I learning about?"))

print("\nTurn 3:")
print(conversation.predict(input="Can you summarize our conversation?"))

In [ ]:
# Inspect memory
print("\n=== Memory Contents ===")
print(memory.load_memory_variables({}))

### Manual Memory Management

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain.memory import ChatMessageHistory

# Create message history
history = ChatMessageHistory()

# Add messages manually
history.add_user_message("What's 2+2?")
history.add_ai_message("2+2 equals 4")
history.add_user_message("What did I just ask?")

# View history
print("Messages:")
for msg in history.messages:
    print(f"{msg.__class__.__name__}: {msg.content}")

# Use with LLM
response = llm.invoke(history.messages)
print(f"\nAI Response: {response.content}")

## 4. Conversation Buffer Window Memory

Only keeps last `k` conversation turns (prevents infinite growth).

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# Keep only last 2 turns (4 messages)
window_memory = ConversationBufferWindowMemory(
    k=2,  # Number of exchanges to remember
    return_messages=True
)

conversation_window = ConversationChain(
    llm=llm,
    memory=window_memory,
    verbose=False
)

# Multiple turns
print("Turn 1:", conversation_window.predict(input="My favorite color is blue"))
print("\nTurn 2:", conversation_window.predict(input="I live in Paris"))
print("\nTurn 3:", conversation_window.predict(input="I love pizza"))

# This should forget turn 1 (favorite color)
print("\nTurn 4 (asking about color - should forget):", 
      conversation_window.predict(input="What's my favorite color?"))

# This should remember (within window)
print("\nTurn 5 (asking about city - should remember):",
      conversation_window.predict(input="Where do I live?"))

## 5. Conversation Summary Memory

Summarizes old messages to save tokens while preserving context.

In [ ]:
from langchain.memory import ConversationSummaryMemory

# Create summary memory
summary_memory = ConversationSummaryMemory(
    llm=llm,
    return_messages=True
)

conversation_summary = ConversationChain(
    llm=llm,
    memory=summary_memory,
    verbose=True
)

# Long conversation
conversation_summary.predict(input="I'm a software engineer working on ML projects")
conversation_summary.predict(input="I recently moved from New York to San Francisco")
conversation_summary.predict(input="I'm interested in learning about LangChain and agents")
conversation_summary.predict(input="I have 5 years of Python experience")

# Check the summary
print("\n=== Memory Summary ===")
print(summary_memory.load_memory_variables({}))

## 6. Conversation Summary Buffer Memory

Combination: keeps recent messages + summarizes old ones.

In [ ]:
from langchain.memory import ConversationSummaryBufferMemory

# Keep recent messages, summarize when exceeds max tokens
summary_buffer_memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=100,  # When exceeded, summarize oldest messages
    return_messages=True
)

conversation_sb = ConversationChain(
    llm=llm,
    memory=summary_buffer_memory,
    verbose=False
)

# Add several turns
turns = [
    "I'm planning a trip to Japan",
    "I want to visit Tokyo, Kyoto, and Osaka",
    "I'm interested in traditional culture and modern technology",
    "My budget is around $3000",
    "I prefer staying in hotels rather than hostels"
]

for turn in turns:
    response = conversation_sb.predict(input=turn)
    print(f"User: {turn}")
    print(f"AI: {response[:100]}...\n")

# Check memory (should have summary + recent messages)
print("\n=== Memory Contents ===")
memory_vars = summary_buffer_memory.load_memory_variables({})
print(f"Type: {type(memory_vars['history'])}")
print(f"Content: {memory_vars}")

## 7. Entity Memory

Tracks specific entities (people, places, organizations) mentioned in conversation.

In [ ]:
from langchain.memory import ConversationEntityMemory

# Create entity memory
entity_memory = ConversationEntityMemory(
    llm=llm,
    return_messages=True
)

conversation_entity = ConversationChain(
    llm=llm,
    memory=entity_memory,
    verbose=True
)

# Conversation mentioning entities
conversation_entity.predict(
    input="Alice works at Google in Mountain View. She's a senior engineer."
)
conversation_entity.predict(
    input="Bob is her colleague. He joined last year from Microsoft."
)

# Ask about entities
print("\n" + "="*50)
print(conversation_entity.predict(input="Tell me about Alice"))
print("\n" + "="*50)
print(conversation_entity.predict(input="Where does Bob work now?"))

# View entity store
print("\n=== Entity Store ===")
print(entity_memory.entity_store.store)

## 8. Vector Store Memory (Semantic Search)

Stores conversation in vector database for semantic similarity search.

In [ ]:
from langchain.memory import VectorStoreRetrieverMemory
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# Create embeddings and vector store
embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_texts(
    ["This is a placeholder"],  # Initial document
    embeddings
)

# Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Create vector memory
vector_memory = VectorStoreRetrieverMemory(
    retriever=retriever,
    memory_key="history",
    input_key="input"
)

# Add some memories
vector_memory.save_context(
    {"input": "I love machine learning and deep learning"},
    {"output": "That's great! What aspect interests you most?"}
)
vector_memory.save_context(
    {"input": "I'm particularly interested in NLP and transformers"},
    {"output": "Transformers are revolutionary! Have you used them?"}
)
vector_memory.save_context(
    {"input": "I also enjoy computer vision tasks"},
    {"output": "CV is fascinating! CNNs or Vision Transformers?"}
)
vector_memory.save_context(
    {"input": "I prefer Python for all my projects"},
    {"output": "Python is excellent for ML development!"}
)

# Semantic search - find relevant past conversations
print("Query: 'What programming language do I use?'")
relevant_memories = vector_memory.load_memory_variables(
    {"input": "What programming language do I use?"}
)
print("\nRelevant memories:")
print(relevant_memories['history'])

print("\n" + "="*50)
print("\nQuery: 'What AI topics am I interested in?'")
relevant_memories = vector_memory.load_memory_variables(
    {"input": "What AI topics am I interested in?"}
)
print("\nRelevant memories:")
print(relevant_memories['history'])

## 9. Persistent Memory (File-Based)

Save memory to disk and reload later.

In [ ]:
import json
from langchain.schema import messages_from_dict, messages_to_dict

# Create a conversation
persistent_memory = ConversationBufferMemory(return_messages=True)
conversation_persist = ConversationChain(llm=llm, memory=persistent_memory)

conversation_persist.predict(input="My name is Alice")
conversation_persist.predict(input="I work as a data scientist")

# Save to file
messages = persistent_memory.chat_memory.messages
messages_dict = messages_to_dict(messages)

with open("conversation_memory.json", "w") as f:
    json.dump(messages_dict, f, indent=2)

print("✅ Memory saved to conversation_memory.json")

# Later: Load from file
with open("conversation_memory.json", "r") as f:
    loaded_messages_dict = json.load(f)

loaded_messages = messages_from_dict(loaded_messages_dict)

# Create new memory with loaded messages
new_memory = ConversationBufferMemory(return_messages=True)
new_memory.chat_memory.messages = loaded_messages

new_conversation = ConversationChain(llm=llm, memory=new_memory)

# Test that memory was restored
print("\nTesting restored memory:")
print(new_conversation.predict(input="What's my name and job?"))

## 10. Memory with LCEL

Using memory with modern LCEL chains.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough

# Store for session histories
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Create prompt with message placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the conversation history to provide context-aware responses."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Create chain
chain = prompt | llm | StrOutputParser()

# Wrap with message history
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Use with session ID
config = {"configurable": {"session_id": "user_123"}}

print("Turn 1:")
print(chain_with_history.invoke(
    {"input": "Hi, I'm learning about memory systems"},
    config=config
))

print("\nTurn 2:")
print(chain_with_history.invoke(
    {"input": "What was I learning about?"},
    config=config
))

# Different session - separate memory
config2 = {"configurable": {"session_id": "user_456"}}

print("\nDifferent user:")
print(chain_with_history.invoke(
    {"input": "What was the previous user learning about?"},
    config=config2
))

## 11. Choosing the Right Memory Type

### Decision Matrix

| Use Case | Best Memory Type | Why |
|----------|------------------|-----|
| **Short conversations** | Buffer Memory | Simple, efficient |
| **Long conversations** | Summary Buffer | Saves tokens, preserves context |
| **Tracking entities** | Entity Memory | Structured entity information |
| **Large knowledge base** | Vector Store | Semantic search over history |
| **Recent context only** | Window Memory | Prevents context overflow |
| **Multi-session** | Persistent + Session ID | Save/load across sessions |

### Token Considerations

```python
# Approximate token usage
Buffer Memory:         # All messages (can be huge!)
Window Memory (k=5):   # Last 10 messages only
Summary Memory:        # 1 summary + recent messages
Summary Buffer:        # Smart hybrid approach
Vector Store:          # Only retrieved relevant messages
```

### Cost vs Context Tradeoff

- **Buffer:** High context, high cost
- **Window:** Medium context, medium cost
- **Summary:** Good context, lower cost
- **Vector:** Relevant context, moderate cost

## 12. Practice Exercise

Build a chatbot that remembers user preferences:

In [ ]:
# Exercise: Create a preference-tracking chatbot
# Requirements:
# 1. Remember user name
# 2. Remember user preferences (favorite food, color, hobby)
# 3. Use entity memory to track these
# 4. Respond appropriately when asked about preferences

# Your code here
# Hint: Use ConversationEntityMemory or create custom storage

# Example solution (uncomment to see)
"""
from langchain.memory import ConversationEntityMemory

preference_memory = ConversationEntityMemory(llm=llm, return_messages=True)
preference_bot = ConversationChain(llm=llm, memory=preference_memory)

# Test conversation
preference_bot.predict(input="My name is Bob, I love pizza and my favorite color is green")
preference_bot.predict(input="I enjoy hiking on weekends")
result = preference_bot.predict(input="What are my favorite things?")
print(result)
"""

## 13. Summary

### Key Takeaways

1. **Memory is essential** for conversational agents
2. **Buffer memory** stores everything (simple but expensive)
3. **Window memory** keeps last k turns (prevents overflow)
4. **Summary memory** compresses old messages (saves tokens)
5. **Entity memory** tracks specific entities (structured info)
6. **Vector memory** enables semantic search (smart retrieval)
7. **Choose based on** conversation length, budget, and requirements
8. **Persistence** allows saving/loading memory across sessions

### What's Next?

In **AG04 - Tools and Function Calling**, you'll learn:
- What are tools in the agent context
- Creating custom tools
- Built-in tools (search, calculator, etc.)
- Function calling with LLMs
- Tool error handling

---

### Quick Quiz

1. **Why do we need memory in agents?**
   <details>
   <summary>Answer</summary>
   LLMs are stateless - they don't remember previous interactions. Memory systems store context to enable multi-turn conversations.
   </details>

2. **When should you use Summary Buffer Memory instead of Buffer Memory?**
   <details>
   <summary>Answer</summary>
   For long conversations where you want to preserve context but save tokens by summarizing old messages.
   </details>

3. **What's the advantage of Vector Store Memory?**
   <details>
   <summary>Answer</summary>
   Semantic search - retrieves only relevant past messages based on similarity, not just recent messages.
   </details>

4. **How do you implement per-user memory in production?**
   <details>
   <summary>Answer</summary>
   Use session IDs to separate memory stores per user, with RunnableWithMessageHistory for LCEL chains.
   </details>

---

**Continue to AG04 to learn about tools! →**